# Substantial Progress Scoring Pipeline
## Federal School Mental Health Grant Programs — Year 1 APR Automation

This notebook automates the **Substantial Progress** determination workflow for two federal school mental health grant programs:

- **MHSP** — Mental Health Service Professionals (higher education grantees)
- **SBMH** — School-Based Mental Health (K–12 education agencies)

Federal Program Officers (FPOs) review each grantee's Annual Performance Report (APR) and assess substantial progress across two dimensions:
1. **Administrative factors** — financial management, staffing stability, internal controls, compliance conditions
2. **GPRA performance** — whether grantees met Year 1 thresholds on each GPRA measure

Traditionally this scoring required FPOs to manually enter 0/1 values into individual Excel workbooks. This pipeline automates steps 2–4 of that workflow.

### What this notebook does
1. **Setup** — configures logging and defines directory paths
2. **Split workbooks** — extracts each FPO's multi-sheet workbook into individual per-grantee `.xlsx` files, preserving all formatting and formulas
3. **Score GPRAs** — loads Year 1 APR data, applies program-specific threshold logic, and produces a structured score dictionary
4. **Insert scores** — writes computed 0/1 GPRA scores into designated cells in each grantee file, adding new rows and updating weighted-average formulas where required

### Data
Input data are synthetic and anonymized for portfolio purposes. The schema mirrors real federal APR Smartsheet exports and FPO workbook templates used by the Department of Education.

---

## 1. Setup

Import libraries, configure logging, and define input/output directory paths.

Update `INPUT_DIR` and `OUTPUT_DIR` to point to your local data folders before running.

In [ ]:
import os
import re
import json
import logging
from copy import copy

import pandas as pd
import numpy as np
from openpyxl import load_workbook, Workbook

# ── Logging ───────────────────────────────────────────────────────────────────
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    filename="notebook_activity.log",
    filemode="w",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
logging.info("Notebook started.")

# ── Directory paths ───────────────────────────────────────────────────────────
# Update these to your local paths before running
DATA_DIR    = os.path.join(os.getcwd(), "data")
INPUT_DIR   = os.path.join(DATA_DIR, "fpo_workbooks")       # Multi-sheet FPO workbooks go here
OUTPUT_DIR  = os.path.join(DATA_DIR, "split_workbooks")     # Per-grantee split files land here
SCORES_DIR  = os.path.join(os.getcwd(), "output")           # Score JSON files

# APR source files
SBMH_APR_PATH = os.path.join(DATA_DIR, "FINAL_Year2_SBMH_APR.xlsx")
MHSP_APR_PATH = os.path.join(DATA_DIR, "FINAL_Year2_MHSP_APR.xlsx")

for d in [OUTPUT_DIR, SCORES_DIR]:
    os.makedirs(d, exist_ok=True)

logging.info("Setup complete.")
print("Setup complete.")

## 2. Split FPO Workbooks into Per-Grantee Files

Each FPO receives a single Excel workbook containing one sheet per assigned grantee. This step splits those multi-sheet workbooks into individual files — one per grantee — named by grant ID (e.g. `S184H240172.xlsx`).

Only sheets whose names contain at least one digit are extracted (this filters out utility sheets like `Directions`, `Definitions`, `SBMH Grantees`, etc.).

Cell formatting, column widths, row heights, and merged cells are all preserved in the output files.

In [ ]:
def sanitize_filename(name: str) -> str:
    """Replace characters not allowed in filenames."""
    return re.sub(r'[\\/*?:"<>|]', '_', name)


def copy_sheet_with_format(source_sheet, target_sheet) -> None:
    """
    Copy all cell values and formatting from source_sheet to target_sheet.
    Also copies column widths, row heights, and merged cell ranges.
    """
    for row in source_sheet.iter_rows():
        for cell in row:
            new_cell = target_sheet[cell.coordinate]
            new_cell.value = cell.value
            if cell.has_style:
                new_cell.font       = copy(cell.font)
                new_cell.border     = copy(cell.border)
                new_cell.fill       = copy(cell.fill)
                new_cell.number_format = copy(cell.number_format)
                new_cell.protection = copy(cell.protection)
                new_cell.alignment  = copy(cell.alignment)

    for col_letter, dim in source_sheet.column_dimensions.items():
        target_sheet.column_dimensions[col_letter].width = dim.width

    for row_idx, dim in source_sheet.row_dimensions.items():
        target_sheet.row_dimensions[row_idx].height = dim.height

    for merged_range in source_sheet.merged_cells.ranges:
        try:
            target_sheet.merge_cells(str(merged_range))
        except Exception as e:
            logging.warning(f"Could not merge cells {merged_range}: {e}")


def split_workbook(file_path: str, output_dir: str) -> int:
    """
    Extract each grantee sheet from a multi-sheet FPO workbook into its own file.

    Only sheets whose name contains at least one digit are extracted.
    Returns the number of files written.
    """
    n_written = 0
    try:
        wb = load_workbook(file_path)
        for sheet_name in wb.sheetnames:
            if not re.search(r'\d', sheet_name):
                logging.info(f"Skipping '{sheet_name}' — no digit in name.")
                continue

            new_wb = Workbook()
            target = new_wb.active
            target.title = sheet_name
            copy_sheet_with_format(wb[sheet_name], target)

            out_path = os.path.join(output_dir, f"{sanitize_filename(sheet_name)}.xlsx")
            new_wb.save(out_path)
            n_written += 1
            logging.info(f"Exported '{sheet_name}' → {out_path}")

        wb.close()
        logging.info(f"Finished processing {os.path.basename(file_path)} ({n_written} sheets exported)")
    except Exception as e:
        logging.error(f"Failed to process {file_path}: {e}")

    return n_written


# ── Process all FPO workbooks ─────────────────────────────────────────────────
total_files = 0
for filename in sorted(os.listdir(INPUT_DIR)):
    if filename.endswith(".xlsx") and not filename.startswith("~$"):
        n = split_workbook(os.path.join(INPUT_DIR, filename), OUTPUT_DIR)
        total_files += n
        print(f"  {filename}: {n} grantee file(s) written")

logging.info(f"Splitting complete. Total files written: {total_files}")
print(f"\nDone. {total_files} grantee files written to: {OUTPUT_DIR}")

## 3. Score GPRA Measures

### 3.1 Load and Clean APR Data

Year 1 APR data are loaded from the MHSP and SBMH source files. Column names are standardized to `snake_case` with a `_y1` suffix. Non-numeric sentinel codes (`999`, `NR`, `N/A`) are replaced with `NaN`.

In [ ]:
def clean_value(x) -> float:
    """
    Coerce a cell value to float.
    Replaces sentinel missing-data codes (999, NR, N/A, NA) with NaN.
    Strips commas from numeric strings before conversion.
    """
    if pd.isna(x):
        return np.nan
    x_str = str(x).strip()
    if re.fullmatch(r'999|n/a|NA|N/A|NR|na', x_str, re.IGNORECASE):
        return np.nan
    try:
        return float(x_str.replace(',', ''))
    except ValueError:
        return np.nan


def convert_ratio(ratio_str) -> float:
    """
    Parse a ratio string (e.g. '49,786/256' or '200:1') into a decimal.
    Returns larger / smaller to normalize direction.
    Handles delimiters: '/', '//', ':', ';'.
    Returns NaN for missing or unparseable input.
    """
    if pd.isna(ratio_str):
        return np.nan
    s = str(ratio_str).replace(',', '').strip()
    for delim in ['//', '/', ':', ';']:
        if s.count(delim) == 1:
            a, b = s.split(delim)
            try:
                num, den = float(a.strip()), float(b.strip())
                if num == 0 or den == 0:
                    return 0.0
                return max(num, den) / min(num, den)
            except ValueError:
                return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan


# ── Load SBMH APR ─────────────────────────────────────────────────────────────
sbmh_df = pd.read_excel(SBMH_APR_PATH, sheet_name=0)
sbmh_df = sbmh_df.rename(columns={
    'Grant ID': 'grant_id', 'Grant Name': 'grantee_name',
    'GPRA 1 (Hired) Target (Raw)':        'gpra_1_target_y2',
    'GPRA 1 (Hired) Actual (Raw)':        'gpra_1_actual_y2',
    'GPRA 2 (Retention) Target (Raw)':    'gpra_2_target_y2',
    'GPRA 2 (Retention) Actual (Raw)':    'gpra_2_actual_y2',
    'GPRA 3 (Ratio) Target':              'gpra_3_target_y2_ratio',
    'GPRA 3 (Ratio) Actual':              'gpra_3_actual_y2_ratio',
    'GPRA 4 (Attrition) Target (Raw)':    'gpra_4_target_y2',
    'GPRA 4 (Attrition) Actual (Raw)':    'gpra_4_actual_y2',
    'GPRA 5 (Students Served) Target (Raw)': 'gpra_5_target_y2',
    'GPRA 5 (Students Served) Actual (Raw)': 'gpra_5_actual_y2',
})
sbmh_df['gpra_3_target_y2'] = sbmh_df['gpra_3_target_y2_ratio'].apply(convert_ratio)
sbmh_df['gpra_3_actual_y2'] = sbmh_df['gpra_3_actual_y2_ratio'].apply(convert_ratio)
for col in ['gpra_1_target_y2', 'gpra_1_actual_y2', 'gpra_2_target_y2', 'gpra_2_actual_y2',
            'gpra_3_target_y2', 'gpra_3_actual_y2', 'gpra_4_target_y2', 'gpra_4_actual_y2',
            'gpra_5_target_y2', 'gpra_5_actual_y2']:
    sbmh_df[col] = sbmh_df[col].apply(clean_value)

# ── Load MHSP APR ─────────────────────────────────────────────────────────────
mhsp_df = pd.read_excel(MHSP_APR_PATH, sheet_name=0)
mhsp_df = mhsp_df.rename(columns={
    'Grant Number': 'grant_id', 'Grant Name': 'grantee_name',
    'GPRA 1A Target (Raw)': 'gpra_1a_target_y2', 'GPRA 1A Actual (Raw)': 'gpra_1a_actual_y2',
    'GPRA 1B Target (Raw)': 'gpra_1b_target_y2', 'GPRA 1B Actual (Raw)': 'gpra_1b_actual_y2',
    'GPRA 2A Target (Raw)': 'gpra_2a_target_y2', 'GPRA 2A Actual (Raw)': 'gpra_2a_actual_y2',
    'GPRA 2B Target (Raw)': 'gpra_2b_target_y2', 'GPRA 2B Actual (Raw)': 'gpra_2b_actual_y2',
    'GPRA 3A Target (Raw)': 'gpra_3a_target_y2', 'GPRA 3A Actual (Raw)': 'gpra_3a_actual_y2',
    'GPRA 3B Target (Raw)': 'gpra_3b_target_y2', 'GPRA 3B Actual (Raw)': 'gpra_3b_actual_y2',
})
for col in ['gpra_1a_target_y2', 'gpra_1a_actual_y2', 'gpra_1b_target_y2', 'gpra_1b_actual_y2',
            'gpra_2a_target_y2', 'gpra_2a_actual_y2', 'gpra_2b_target_y2', 'gpra_2b_actual_y2',
            'gpra_3a_target_y2', 'gpra_3a_actual_y2', 'gpra_3b_target_y2', 'gpra_3b_actual_y2']:
    mhsp_df[col] = mhsp_df[col].apply(clean_value)

print(f"SBMH loaded: {sbmh_df.shape[0]} grantees")
print(f"MHSP loaded: {mhsp_df.shape[0]} grantees")
logging.info(f"APR data loaded. SBMH: {sbmh_df.shape[0]} rows, MHSP: {mhsp_df.shape[0]} rows.")

### 3.2 Apply Scoring Logic

Each GPRA measure has a program-specific threshold that defines "substantial progress" in Year 1. Scores are binary: **1** = threshold met, **0** = threshold not met or data missing.

**MHSP thresholds (Year 1):**

| Measure | Threshold |
|---------|----------|
| GPRA 1a (Trained, annual) | actual ≥ 0% of target |
| GPRA 1b (Placed, current) | actual ≥ 0% of target |
| GPRA 2a (Internship, annual) | actual ≥ 5% of target |
| GPRA 2b (Internship, current) | actual ≥ 5% of target |
| GPRA 3a (Retained, annual) | actual ≥ 0% of target |
| GPRA 3b (Retained, current) | actual ≥ 0% of target |

**SBMH thresholds (Year 1):**

| Measure | Threshold |
|---------|----------|
| GPRA 1 (Hired) | actual ≥ 20% of target |
| GPRA 2 (Retained) | actual ≥ 20% of target |
| GPRA 3 (Student ratio) | actual ratio ≤ target ratio |
| GPRA 4 (Served via telehealth) | actual ≥ 11% of target |
| GPRA 5 (Students served) | actual ≥ target |

> **Note:** Year 1 thresholds reflect early-grant-period expectations — grantees are not expected to fully meet multi-year targets in their first reporting period.

In [ ]:
def score_mhsp(row: pd.Series) -> dict:
    """
    Score a single MHSP grantee row against Year 1 GPRA thresholds.
    Returns a dict of {measure: 0_or_1}.
    Missing data (NaN in actual or target) scores as 0.
    """
    def meets(actual_col, target_col, pct):
        a, t = row.get(actual_col), row.get(target_col)
        if pd.isna(a) or pd.isna(t):
            return 0
        return int(a >= pct * t)

    return {
        'GPRA1a': meets('gpra_1a_actual_y2', 'gpra_1a_target_y2', 0.00),
        'GPRA1b': meets('gpra_1b_actual_y2', 'gpra_1b_target_y2', 0.00),
        'GPRA2a': meets('gpra_2a_actual_y2', 'gpra_2a_target_y2', 0.05),
        'GPRA2b': meets('gpra_2b_actual_y2', 'gpra_2b_target_y2', 0.05),
        'GPRA3a': meets('gpra_3a_actual_y2', 'gpra_3a_target_y2', 0.00),
        'GPRA3b': meets('gpra_3b_actual_y2', 'gpra_3b_target_y2', 0.00),
    }


def score_sbmh(row: pd.Series) -> dict:
    """
    Score a single SBMH grantee row against Year 1 GPRA thresholds.
    GPRA 3 uses ratio comparison (lower is better).
    Returns a dict of {measure: 0_or_1}.
    """
    def meets(actual_col, target_col, pct):
        a, t = row.get(actual_col), row.get(target_col)
        if pd.isna(a) or pd.isna(t):
            return 0
        return int(a >= pct * t)

    def meets_ratio(actual_col, target_col):
        a, t = row.get(actual_col), row.get(target_col)
        if pd.isna(a) or pd.isna(t):
            return 0
        return int(a <= t)  # lower ratio = better

    return {
        'GPRA1': meets('gpra_1_actual_y2', 'gpra_1_target_y2', 0.20),
        'GPRA2': meets('gpra_2_actual_y2', 'gpra_2_target_y2', 0.20),
        'GPRA3': meets_ratio('gpra_3_actual_y2', 'gpra_3_target_y2'),
        'GPRA4': meets('gpra_4_actual_y2', 'gpra_4_target_y2', 0.11),
        'GPRA5': meets('gpra_5_actual_y2', 'gpra_5_target_y2', 1.00),
    }


# ── Score all grantees ────────────────────────────────────────────────────────
mhsp_scores = {
    str(row.get('grant_id', f'MHSP_{idx}')): score_mhsp(row)
    for idx, row in mhsp_df.iterrows()
}
sbmh_scores = {
    str(row.get('grant_id', f'SBMH_{idx}')): score_sbmh(row)
    for idx, row in sbmh_df.iterrows()
}

# Save intermediate score files for audit and reproducibility
os.makedirs(SCORES_DIR, exist_ok=True)
with open(os.path.join(SCORES_DIR, 'mhsp_scores.json'), 'w') as f:
    json.dump(mhsp_scores, f, indent=2)
with open(os.path.join(SCORES_DIR, 'sbmh_scores.json'), 'w') as f:
    json.dump(sbmh_scores, f, indent=2)

logging.info(f"Scored {len(mhsp_scores)} MHSP and {len(sbmh_scores)} SBMH grantees.")
print(f"Scored {len(mhsp_scores)} MHSP grantees and {len(sbmh_scores)} SBMH grantees.")
print(f"Score files saved to: {SCORES_DIR}")

## 4. Insert Scores into Grantee Files

This step writes computed 0/1 GPRA scores into designated cells in each per-grantee `.xlsx` file. It also:

- **SBMH files**: corrects a formula sign issue in row 23, and enforces integer formatting in row 18
- **MHSP files**: inserts two new rows after row 26 for GPRA 3a/3b (measures not present in the original Year 1 template), copies row formatting from row 23, and updates five weighted-average formulas that now include the new GPRA 3 rows

All modifications are logged. Files that cannot be matched to a grant ID in the score dictionaries are skipped with a warning.

In [ ]:
# ── Cell mappings: GPRA measure → workbook cell ───────────────────────────────
MHSP_CELL_MAP = {
    'GPRA1a': 'G19',
    'GPRA1b': 'G20',
    'GPRA2a': 'G23',
    'GPRA2b': 'G24',
    'GPRA3a': 'G27',  # inserted row
    'GPRA3b': 'G28',  # inserted row
}
SBMH_CELL_MAP = {
    'GPRA1': 'G18',
    'GPRA2': 'G19',
    'GPRA3': 'G20',
    'GPRA4': 'G21',
    'GPRA5': 'G22',
}


def patch_sbmh_file(ws) -> None:
    """Apply SBMH-specific formula and formatting corrections."""
    # Fix formula sign issue: =-expression → =expression
    for cell_ref in ['E23', 'G23', 'I23', 'K23', 'M23']:
        val = ws[cell_ref].value
        if isinstance(val, str) and val.startswith('=-'):
            ws[cell_ref].value = '=' + val[2:]
    # Enforce integer formatting in row 18
    for col in ['E', 'G', 'I', 'K', 'M']:
        cell_obj = ws[f'{col}18']
        if cell_obj.number_format != '0':
            cell_obj.number_format = '0'


def patch_mhsp_file(ws, scores: dict) -> None:
    """Insert GPRA 3a/3b rows and update weighted-average formulas for MHSP files."""
    # Insert 2 rows after row 26 to accommodate GPRA 3a and 3b
    ws.insert_rows(27, amount=2)

    # Copy formatting from row 23 template row to new rows 27 and 28
    for col in range(1, ws.max_column + 1):
        src = ws.cell(row=23, column=col)
        for tgt_row in [27, 28]:
            tgt = ws.cell(row=tgt_row, column=col)
            if src.has_style:
                tgt._style = src._style

    # Label new rows
    ws['A27'] = 'GPRA 3A: Annual'
    ws['A28'] = 'GPRA 3B: Current'
    ws['B26'] = ''
    ws['B27'] = 0
    ws['B28'] = 0

    # Update GPRA score row (row 29) formulas to include new measures
    ws['B29'] = '=SUM(B19*0.2, B20*0.4, B23*0.1, B24*0.1, B27*0.1, B28*0.1)'
    ws['E29'] = '=SUM(E19*0.2, E20*0.4, E23*0.1, E24*0.1, E27*0.1, E28*0.1)'
    ws['G29'] = '=SUM(G19*0.2, G20*0.4, G23*0.1, G24*0.1, G27*0.10, G28*0.10)'
    ws['I29'] = '=SUM(I19*0.3, I20*0.2, I23*0.2, I24*0.1, I27*0.10, I28*0.10)'
    ws['K29'] = '=SUM(K19*0.2, K20*0.2, K23*0.2, K24*0.2, K27*0.10, K28*0.10)'
    ws['M29'] = '=SUM(M19*0.1, M20*0.1, M23*0.2, M24*0.2, M27*0.20, M28*0.20)'

    # Update overall score row (row 32) formulas
    ws['C32'] = '=SUM(C15*0.6, B29*0.4)'
    ws['E32'] = '=SUM(E15*0.6, E29*0.4)'
    ws['G32'] = '=SUM(G15*0.6, G29*0.4)'
    ws['I32'] = '=SUM(I15*0.5, I29*0.5)'
    ws['K32'] = '=SUM(K15*0.4, K29*0.6)'
    ws['M32'] = '=SUM(M15*0.3, M29*0.7)'


# ── Main loop ─────────────────────────────────────────────────────────────────
log_entries = []
n_updated, n_skipped = 0, 0

for filename in sorted(os.listdir(OUTPUT_DIR)):
    if not filename.endswith('.xlsx') or filename.startswith('~$'):
        continue

    grant_id = filename.replace('.xlsx', '')

    if grant_id in mhsp_scores:
        scores, cell_map, is_mhsp = mhsp_scores[grant_id], MHSP_CELL_MAP, True
    elif grant_id in sbmh_scores:
        scores, cell_map, is_mhsp = sbmh_scores[grant_id], SBMH_CELL_MAP, False
    else:
        msg = f"Skipped {filename}: no score entry found."
        log_entries.append(msg)
        logging.warning(msg)
        n_skipped += 1
        continue

    file_path = os.path.join(OUTPUT_DIR, filename)
    wb = load_workbook(file_path)
    ws = wb.worksheets[0]

    if is_mhsp:
        patch_mhsp_file(ws, scores)
    else:
        patch_sbmh_file(ws)

    # Write 0/1 scores into mapped cells
    for gpra, value in scores.items():
        if gpra in cell_map:
            ws[cell_map[gpra]] = value

    wb.save(file_path)
    wb.close()

    msg = f"Updated {filename} ({grant_id})."
    log_entries.append(msg)
    logging.info(msg)
    n_updated += 1

# Save run log
log_path = os.path.join(SCORES_DIR, 'update_log.txt')
with open(log_path, 'w') as f:
    f.write('\n'.join(log_entries))

logging.info(f"Score insertion complete. Updated: {n_updated}, Skipped: {n_skipped}.")
print(f"Score insertion complete.")
print(f"  Updated : {n_updated} files")
print(f"  Skipped : {n_skipped} files (no score entry)")
print(f"  Run log : {log_path}")